# Module 4 — Developing your first AI Agent in Azure AI Foundry

> Part of the **"Develop & Deploy AI Agents on Azure with LangChain, Python and Foundry"** course.

In Module 3 we wrote an agent ourselves, in Python, with LangChain. Foundry offers a different option: **let Azure host the agent for you**.

## 🎯 Learning objectives

1. Understand what an **Azure AI Foundry Agent** (Persistent Agent) is and how it differs from a LangChain agent.
2. Create an agent **declaratively** with the `azure-ai-agents` SDK.
3. Start a **thread**, send messages, and **stream** the run.
4. Inspect the **server-side trace** of every step in the Foundry portal.
5. Decide when to pick **Foundry-hosted** vs **LangChain-hosted** agents.

## 🧠 Key concepts

| Concept                 | What it means                                                                            |
| ----------------------- | ---------------------------------------------------------------------------------------- |
| **Persistent Agent**    | An agent definition (instructions + tools + model) stored *server-side* in Foundry.      |
| **Thread**              | A conversation. Each user gets one. Messages and runs are nested under a thread.         |
| **Run**                 | One execution of the agent loop on a thread, started by `runs.create_and_process(...)`.  |
| **Server-side tools**   | Built-in tools Foundry runs for you: Code Interpreter, File Search, Bing, Logic Apps…    |
| **Function tools**      | Tools whose implementation lives in **your** code (the SDK passes the call back to you). |

## 🆚 LangChain agent vs Foundry agent

| Aspect                 | LangChain (Module 3)                       | Foundry Persistent Agent                       |
| ---------------------- | ------------------------------------------ | ---------------------------------------------- |
| Where it runs          | Your process / container                   | Managed Azure service                          |
| State / memory         | You manage it (Module 11)                  | Threads are stored automatically               |
| Built-in tools         | None — you wire them                       | Code Interpreter, File Search, Bing, Logic Apps |
| Custom tools           | Native Python                              | Pass tool definitions, implement callbacks     |
| Tracing                | LangSmith / OTel (Module 14)               | Built-in in the Foundry portal                 |
| Portability            | Run anywhere Python runs                   | Azure-only                                     |

👉 You'll often **combine** them: LangChain on the outside, Foundry-hosted models on the inside, and Foundry agents reserved for ops-heavy workflows that benefit from managed state.

## 📋 Prerequisites

* Module 2a complete (Foundry project + model deployment exist).
* `az login` (the SDK authenticates with `DefaultAzureCredential`).
* Install the SDK:

In [ ]:
%pip install azure-ai-agents azure-identity python-dotenv

## 🔌 Read the Foundry endpoint from Terraform

The Foundry **project endpoint** (different from the model endpoint!) is what the Agents SDK consumes.

In [ ]:
foundry_project_endpoint = ! terraform -chdir=./infra output -raw foundry_project_endpoint
foundry_project_endpoint = foundry_project_endpoint.n
print("Project endpoint :", foundry_project_endpoint)

llm_model_deployment_name_chatgpt = ! terraform -chdir=./infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("Model deployment :", llm_model_deployment_name_chatgpt)

## 🏗️ Step 1 — Create the agent

The agent definition is **persistent**: it survives process restarts. Re-running this cell with the same name will create a *new* agent — store the returned ID if you want to reuse it.

In [ ]:
from azure.ai.agents import AgentsClient
from azure.identity import DefaultAzureCredential

client = AgentsClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

agent = client.create_agent(
    model=llm_model_deployment_name_chatgpt,
    name="course-first-foundry-agent",
    instructions=(
        "You are a friendly Azure expert. "
        "Answer concisely and always suggest the most cost-effective option."
    ),
)

print("Created agent id:", agent.id)

## 💬 Step 2 — Create a thread and send a message

Think of the thread as one user's chat session.

In [ ]:
thread = client.threads.create()
print("Thread id:", thread.id)

client.messages.create(
    thread_id=thread.id,
    role="user",
    content="What's the cheapest way to host a small AI agent on Azure?",
)

## ▶️ Step 3 — Run the agent and read the reply

`create_and_process` is the convenient one-liner that submits a run, polls until it's done, and returns when the agent has finished thinking.

In [ ]:
run = client.runs.create_and_process(
    thread_id=thread.id,
    agent_id=agent.id,
)
print("Run status:", run.status)

for msg in client.messages.list(thread_id=thread.id, order="asc"):
    for part in msg.content:
        if getattr(part, "text", None):
            print(f"\n[{msg.role}]\n{part.text.value}")

## 🧹 Step 4 — Clean up

Foundry charges for stored agents and threads. Delete what you don't need in the lab — production agents are normally created once and reused.

In [ ]:
client.threads.delete(thread.id)
client.delete_agent(agent.id)
print("Cleaned up ✅")

## 🧭 Where next

* **Module 5** — add custom tools to a LangChain agent.
* **Module 11** — when you need cross-thread memory in LangChain, Foundry already gives it for free.
* The Foundry portal shows a step-by-step trace of every run — open it now and confirm what just happened.